# **Environment Setup**

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import wandb
from kaggle_secrets import UserSecretsClient

import torch
from datasets import load_dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer)

2025-12-28 09:28:35.659516: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766914115.841422      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766914115.898262      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766914116.336276      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766914116.336313      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766914116.336316      55 computation_placer.cc:177] computation placer alr

In [3]:
args = TrainingArguments(
    output_dir="roberta-base-fft",
    
    # Training Hyperparameters
    num_train_epochs = 2,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 1,
    learning_rate = 5e-5,
    
    # Optimization & Precision
    optim = "adamw_torch",
    bf16 = True,
    logging_steps = 500,
    gradient_checkpointing = True,
    
    # Scheduling
    lr_scheduler_type = "cosine",
    warmup_steps = 100,
    
    # Logging & Reporting
    report_to = "wandb",
    disable_tqdm = False,

    # Saving Options
    save_safetensors = False,
    save_total_limit = 2
)

In [4]:
config_dict = {
    "model_name": "FacebookAI/xlm-roberta-base",
    "per_device_train_batch_size": 4,
    "gradient_accumulation_steps": 1,
    "num_train_epochs": 2,
    "learning_rate": 5e-5,
    "optim": "adamw_torch",
    "bf16": True,
    "gradient_checkpointing": True,
    "tags": ["FFT", "Part-I", "xlm-roberta-base"]
}

In [5]:
user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key = wandb_api_key)

wandb.init(
    project = "sft-nlp-4",
    config = config_dict,
    name = "roberta-fft-baseline",
    tags = config_dict["tags"] 
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mohassan5286 (mohassan5286-alexandria-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# **Getting Model & Tokenizer**

In [6]:
# Load Model
model_id = 'FacebookAI/xlm-roberta-base'
model = AutoModelForCausalLM.from_pretrained(model_id, device_map = 'cuda')
model.config.is_decoder = True
model.config.add_cross_attention = False

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    use_fast        = True,
    truncation_side = "right",
    padding_side    = "right",
    add_eos_token   = True,
    add_bos_token   = True,
)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

If you want to use `XLMRobertaLMHeadModel` as a standalone, add `is_decoder=True.`


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# **Load Dataset**

In [7]:
dataset = load_dataset("flytech/python-codes-25k", split = "train")
train_dataset = dataset.shuffle(seed = 42).select(range(38_000))

print(f"Training samples: {len(train_dataset)}")

README.md: 0.00B [00:00, ?B/s]

python-codes-25k.json:   0%|          | 0.00/26.4M [00:00<?, ?B/s]

python-codes-25k.jsonl:   0%|          | 0.00/25.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49626 [00:00<?, ? examples/s]

Training samples: 38000


# **Tokenize Dataset**

In [8]:
def tokenize_function(examples):
    texts = []
    instr_lens = []

    for instr, out in zip(examples["instruction"], examples["output"]):
        instr_text = f"{tokenizer.bos_token}{instr}\nAnswer:\n"
        full_text = instr_text + out + tokenizer.eos_token
        texts.append(full_text)

        instr_ids = tokenizer(
            instr_text,
            truncation = True,
            max_length = 512,
            add_special_tokens = False
        )["input_ids"]
        instr_lens.append(len(instr_ids))

    tokenized = tokenizer(
        texts,
        truncation = True,
        max_length = 512,
        padding = "max_length"
    )

    labels = []
    for ids, l in zip(tokenized["input_ids"], instr_lens):
        l = min(l, 512)
        label = [-100] * l + ids[l:]
        label = [
            -100 if t == tokenizer.pad_token_id else t
            for t in label
        ]
        labels.append(label[:512])

    tokenized["labels"] = labels
    return tokenized

In [9]:
train_dataset = train_dataset.map(tokenize_function, batched = True, remove_columns = train_dataset.column_names)

Map:   0%|          | 0/38000 [00:00<?, ? examples/s]

# **Training**

In [10]:
trainer = Trainer(
    model = model,
    args = args,
    train_dataset = train_dataset
)

trainer.train()

Step,Training Loss
500,4.305500
1000,2.637300
1500,2.256400
2000,2.114600
2500,1.927300
3000,1.850600
3500,1.759200
4000,1.682600
4500,1.681800
5000,1.587900


TrainOutput(global_step=19000, training_loss=1.4703424361379522, metrics={'train_runtime': 38656.9312, 'train_samples_per_second': 1.966, 'train_steps_per_second': 0.492, 'total_flos': 2.005480820736e+16, 'train_loss': 1.4703424361379522, 'epoch': 2.0})

# **Evaluation**

In [11]:
input_text = "Create a shopping list based on my inputs!"
prompt = f"{tokenizer.bos_token}{input_text}\nAnswer:\n"

inputs = tokenizer(prompt, return_tensors="pt").to('cuda')
    
model.eval()
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens = 64,
        do_sample = True,
        temperature = 0.6,
        repetition_penalty = 1.2,
        pad_token_id = tokenizer.eos_token_id,
        eos_token_id = tokenizer.eos_token_id
    )

decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

if "Answer:" in decoded:
    print(decoded.split("Answer:", 1)[1].strip())

else:
    print(decoded.strip())

```python def sort_by_price(arr): n = len(5, 3) m1 = 0 for i in range(1 row+1)-1 while (m1) <= 1: if arr[i] > max and j % 2 !=0 : minValueError occur
